# Import

In [ ]:
pip install tensorly scipy scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

from sklearn.base import clone

from tensorly.decomposition import parafac
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)
from sklearn.model_selection import (
    GroupKFold,
    KFold,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

print("XGBoost available:", HAS_XGBOOST)

# Data loading

In [ ]:
import gdown
url = "https://drive.google.com/drive/folders/1xUqTc9Dk0Gexo7Kcu5aa8hKfxWL2tShI?usp=sharing"
gdown.download_folder(
    url,
    output="dataset",
    quiet=False
)

In [ ]:
import os

os.listdir("dataset")

In [ ]:
# ---------- Data ----------
EEM_PATH = Path("dataset/eem.npy")
SAMPLES_PATH = Path("dataset/samples.parquet")
WAVELENGTH_PATH = Path("dataset/wavelengths.npz")

# ---------- Cross-validation ----------
# "group": evaluate generalization to unseen sampling locations
# "random": ordinary shuffled KFold
CV_MODE = "group"
GROUP_COL = "Point"   # alternatively: "label"
OUTER_SPLITS = 5
INNER_SPLITS = 3
RANDOM_STATE = 42

# ---------- Runtime ----------
# Quick grids are deliberately small for first benchmarking.
RUN_MODE = "quick"   # currently only quick grids are defined
N_JOBS = -1

# Set to None to run every available experiment.
MODELS_TO_RUN = None

# Set to None after loading to run all detected water-quality targets.
# Example: TARGETS_TO_RUN = ["TOC\n(0.0)", "NH3-N\n(0.000)"]
TARGETS_TO_RUN = None

# Optional: strongly right-skewed targets can be modeled in log1p space.
# Predictions are transformed back before evaluation.
# Keep empty for the first benchmark; then compare raw vs log-target experiments.
LOG1P_TARGETS = set()

PH_COL = "pH(0.0)"
TEMP_COL = "Temp(0.0)"
MONTH_COL = "month"
SS_COL = "SS\n(0.0)"
EC_COL = "EC\n(0)"
BOD_COL = "BOD\n(0.0)"
COD_COL = "COD\n(0.0)"
TOC_COL = "TOC\n(0.0)"

In [ ]:
eem = np.load(EEM_PATH)
samples = pd.read_parquet(SAMPLES_PATH)

assert eem.ndim == 3, f"Expected EEM shape (samples, ex, em), got {eem.shape}"
assert len(eem) == len(samples), "EEM and metadata sample counts do not match."

n_samples, n_ex, n_em = eem.shape

print("EEM shape:", eem.shape)
print("Metadata shape:", samples.shape)
print("Pixels per EEM:", n_ex * n_em)
print("Finite EEM fraction:", np.isfinite(eem).mean())

display(samples.head())

In [ ]:
import numpy as np

# eem shape: (765, 57, 271)
eem = eem.astype(np.float32)

# ==========================================
# 1. Apply 2D FFT to each EEM matrix
# ==========================================

fft = np.fft.fft2(
    eem,
    axes=(-2, -1)
)

# Move low frequencies to the center
fft = np.fft.fftshift(
    fft,
    axes=(-2, -1)
)

# FFT is complex -> use magnitude
fft_mag = np.abs(fft)

# Log transform to reduce very large dynamic range
fft_mag = np.log1p(
    fft_mag
).astype(np.float32)


# ==========================================
# 2. Stack original EEM + FFT
# ==========================================

# eem:
# (N, 57, 271)
#
# fft_mag:
# (N, 57, 271)

X_eem_fft = np.stack(
    [
        eem,
        fft_mag
    ],
    axis=1
)

print("Original EEM:", eem.shape)
print("FFT:", fft_mag.shape)
print("2-channel input:", X_eem_fft.shape)

In [ ]:
samples["BOD_COD"] = samples["BOD\n(0.0)"] / samples["COD\n(0.0)"]

In [ ]:
# Known water-quality columns from this dataset.
TARGET_CANDIDATES = [
    "BOD_COD"
    # "BOD\n(0.0)",
    # "COD\n(0.0)",
    # "TOC\n(0.0)",
    # "SS\n(0.0)",
    # "EC\n(0)",
    # "T-N\n(0.000)",
    # "DTN\n(0.000)",
    # "NO3-N\n(0.000)",
    # "NH3-N\n(0.000)",
    # "T-P\n(0.000)",
    # "DTP\n(0.000)",
    # "PO₄-P\n(0.000)",
    # "Chlorophyll a\n(0.0)",
    # "E-coliform group(0)",
    # "Fecal coliform count(0)",
]

target_cols = [c for c in TARGET_CANDIDATES if c in samples.columns]

# Fallback: show numeric columns if names differ from the candidate list.
if not target_cols:
    metadata_numeric = {"sample_id", "round", "month", "Year", "Month"}
    target_cols = [
        c for c in samples.select_dtypes(include=np.number).columns
        if c not in metadata_numeric
    ]

if TARGETS_TO_RUN is None:
    TARGETS_TO_RUN = target_cols
else:
    missing = [c for c in TARGETS_TO_RUN if c not in samples.columns]
    if missing:
        raise KeyError(f"Targets not found: {missing}")

print("Detected targets:")
for c in target_cols:
    print(" -", repr(c))

summary = pd.DataFrame({
    "n_valid": samples[TARGETS_TO_RUN].notna().sum(),
    "missing_%": samples[TARGETS_TO_RUN].isna().mean() * 100,
    "median": samples[TARGETS_TO_RUN].median(numeric_only=True),
    "mean": samples[TARGETS_TO_RUN].mean(numeric_only=True),
})
display(summary)

In [ ]:
import matplotlib.pyplot as plt

param = "BOD\n(0.0)"   # sửa theo tên cột của bạn

months = sorted(samples["month"].dropna().unique())

data = [
    samples.loc[samples["month"] == m, param].dropna()
    for m in months
]

plt.figure(figsize=(10, 5))
plt.boxplot(data, labels=months)

plt.xlabel("Month")
plt.ylabel(param)
plt.title(f"{param} distribution by month")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import kruskal

PARAMS = [
    "BOD\n(0.0)",
    "COD\n(0.0)",
    "TOC\n(0.0)",
    "SS\n(0.0)",
    "EC\n(0)",
]

results = []

for param in PARAMS:

    df = samples[
        ["month", param]
    ].dropna()

    groups = [
        group[param].values
        for _, group in df.groupby("month")
        if len(group) > 0
    ]

    stat, p = kruskal(*groups)

    results.append({
        "parameter": param,
        "Kruskal_stat": stat,
        "p_value": p,
        "significant": p < 0.05
    })

month_effect_df = pd.DataFrame(results)

month_effect_df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# List of parameters to analyze
PARAMS = [
    "BOD\n(0.0)",
    "COD\n(0.0)",
    "TOC\n(0.0)",
    "SS\n(0.0)",
    "EC\n(0)",
    "Temp(0.0)",
    "pH(0.0)",
    # "T-N\n(0.000)",
    # "DTN\n(0.000)",
    # "NO3-N\n(0.000)",
    # "NH3-N\n(0.000)",
    # "T-P\n(0.000)",
    # "DTP\n(0.000)",
    # "PO₄-P\n(0.000)",
    # "Chlorophyll a\n(0.0)",
    # "E-coliform group(0)",
    # "Fecal coliform count(0)",
]

# Select parameters
df_corr = samples[PARAMS].copy()

# Correlation matrix
corr_matrix = df_corr.corr(method="spearman")

print(corr_matrix)

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(
    corr_matrix,
    vmin=-1,
    vmax=1
)

ax.set_xticks(range(len(PARAMS)))
ax.set_yticks(range(len(PARAMS)))

ax.set_xticklabels(PARAMS, rotation=45, ha="right")
ax.set_yticklabels(PARAMS)

# Add correlation values
for i in range(len(PARAMS)):
    for j in range(len(PARAMS)):
        value = corr_matrix.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center"
        )

fig.colorbar(im, ax=ax, label="Pearson correlation")

ax.set_title("Spearman Correlation Matrix of Water Quality Parameters")

plt.tight_layout()
plt.show()

In [ ]:
param = "pH(0.0)"

variance = samples[param].var()

print("Variance:", variance)

In [ ]:
# Flatten EEM: (n_samples, n_ex, n_em) -> (n_samples, n_pixels)
# Convert inf/-inf to NaN so sklearn's SimpleImputer can handle them.
X_eem = eem.reshape(n_samples, -1).astype(np.float64, copy=True)
X_eem[~np.isfinite(X_eem)] = np.nan

print("Flattened EEM:", X_eem.shape)
print("Approx. size (MB):", X_eem.nbytes / 1024**2)

In [ ]:
samples.columns

# Experiments

## ML models

In [ ]:
def fit_parafac_and_project(
    X_train,
    X_test,
    n_components=4,
    n_iter_max=1000,
    tol=1e-7,
    random_state=42
):
    """
    X_train: (N_train, n_ex, n_em)
    X_test : (N_test,  n_ex, n_em)

    Returns
    -------
    train_scores : (N_train, R)
    test_scores  : (N_test, R)
    ex_loadings  : (n_ex, R)
    em_loadings  : (n_em, R)
    cp_model
    """

    # ==========================================
    # Fit PARAFAC ONLY on training data
    # ==========================================

    cp_model = parafac(
        X_train,
        rank=n_components,
        init="svd",
        n_iter_max=n_iter_max,
        tol=tol,
        random_state=random_state,
        normalize_factors=False
    )

    weights, factors = cp_model

    # factors:
    # A: sample mode
    # B: excitation mode
    # C: emission mode
    A_train, B_ex, C_em = [
        np.asarray(f)
        for f in factors
    ]

    # ==========================================
    # Absorb CP weights into sample scores
    # ==========================================

    if weights is not None:
        weights = np.asarray(weights)

        A_train = (
            A_train
            * weights.reshape(1, -1)
        )

    # ==========================================
    # Construct fixed component basis
    #
    # component_r =
    # outer(Ex_loading_r, Em_loading_r)
    # ==========================================

    basis = np.column_stack([
        np.outer(
            B_ex[:, r],
            C_em[:, r]
        ).reshape(-1)

        for r in range(n_components)
    ])

    # basis shape:
    # (n_ex * n_em, R)

    # ==========================================
    # Project TEST EEM onto TRAIN components
    # ==========================================

    X_test_flat = X_test.reshape(
        X_test.shape[0],
        -1
    )

    # Solve:
    #
    # basis @ score_i ≈ eem_i
    #
    # all test samples simultaneously

    test_scores = np.linalg.lstsq(
        basis,
        X_test_flat.T,
        rcond=None
    )[0].T

    # ==========================================
    # Reconstruction diagnostics
    # ==========================================

    X_train_flat = X_train.reshape(
        X_train.shape[0],
        -1
    )

    train_reconstructed = (
        A_train @ basis.T
    )

    test_reconstructed = (
        test_scores @ basis.T
    )

    train_error = (
        np.linalg.norm(
            X_train_flat - train_reconstructed
        )
        /
        np.linalg.norm(X_train_flat)
    )

    test_error = (
        np.linalg.norm(
            X_test_flat - test_reconstructed
        )
        /
        np.linalg.norm(X_test_flat)
    )

    print(
        f"PARAFAC rank={n_components} | "
        f"train relative error={train_error:.4f} | "
        f"test relative error={test_error:.4f}"
    )

    return (
        A_train,
        test_scores,
        B_ex,
        C_em,
        cp_model
    )

In [ ]:
# =========================================================
# PREPARE EEM
# =========================================================

X_eem_parafac = np.asarray(
    eem,
    dtype=float
)

# Nếu dữ liệu có shape:
# (N, 1, H, W)
# -> (N, H, W)

if (
    X_eem_parafac.ndim == 4
    and X_eem_parafac.shape[1] == 1
):
    X_eem_parafac = X_eem_parafac[:, 0]


if X_eem_parafac.ndim != 3:
    raise ValueError(
        "PARAFAC requires EEM with shape "
        "(N, n_ex, n_em)"
    )


if not np.isfinite(X_eem_parafac).all():
    raise ValueError(
        "X_eem contains NaN or inf. "
        "Please preprocess EEM before PARAFAC."
    )


# =========================================================
# GLOBAL TRAIN / TEST SPLIT
# =========================================================

all_idx = np.arange(
    len(samples)
)

train_idx_global, test_idx_global = train_test_split(
    all_idx,
    test_size=0.2,
    random_state=42
)

print(
    "Global train:",
    len(train_idx_global),
    "| Global test:",
    len(test_idx_global)
)

In [ ]:
# =========================================================
# PARAFAC CONFIG
# =========================================================

PARAFAC_N_COMPONENTS =4


# =========================================================
# FIT PARAFAC ON TRAIN ONLY
# =========================================================

X_eem_train_pf = X_eem_parafac[
    train_idx_global
]

X_eem_test_pf = X_eem_parafac[
    test_idx_global
]


(
    X_parafac_train,
    X_parafac_test,
    parafac_ex_loadings,
    parafac_em_loadings,
    parafac_model
) = fit_parafac_and_project(
    X_eem_train_pf,
    X_eem_test_pf,
    n_components=PARAFAC_N_COMPONENTS
)

print(
    "PARAFAC train features:",
    X_parafac_train.shape
)

print(
    "PARAFAC test features:",
    X_parafac_test.shape
)

In [ ]:
# =========================================================
# MAP PARAFAC FEATURES BACK TO ORIGINAL SAMPLE INDICES
# =========================================================

parafac_features = np.full(
    (
        len(samples),
        PARAFAC_N_COMPONENTS
    ),
    np.nan
)

parafac_features[
    train_idx_global
] = X_parafac_train

parafac_features[
    test_idx_global
] = X_parafac_test

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================
# Config
# =============================
TARGETS = [BOD_COL, COD_COL, TOC_COL]   # sửa theo dataset
n_components = parafac_features.shape[1]

# PARAFAC -> DataFrame
parafac_df = pd.DataFrame(
    parafac_features,
    columns=[f"PARAFAC_{i+1}" for i in range(n_components)],
    index=samples.index
)

# Ghép target + PARAFAC features
df_corr = pd.concat(
    [samples[TARGETS], parafac_df],
    axis=1
)

# =============================
# Pearson correlation
# =============================
corr = df_corr.corr(method="spearman")

# chỉ lấy Target x PARAFAC
target_parafac_corr = corr.loc[
    TARGETS,
    parafac_df.columns
]

print(target_parafac_corr)

In [ ]:
EXPERIMENTS = [

    # =========================================================
    # Tabular only
    # =========================================================
    {
        "name": "TOC",
        "use_eem": False,
        "use_parafac": False,
        "features": [
            TOC_COL
        ]
    },

    {
        "name": "TOC_SS_EC",
        "use_eem": False,
        "use_parafac": False,
        "features": [
            TOC_COL,
            SS_COL,
            EC_COL,
        ]
    },

    # =========================================================
    # PARAFAC
    # =========================================================
    {
        "name": "PARAFAC",
        "use_eem": False,
        "use_parafac": True,
        "features": []
    },

    {
        "name": "PARAFAC_TOC",
        "use_eem": False,
        "use_parafac": True,
        "features": [
            TOC_COL
        ]
    },

    {
        "name": "PARAFAC_SS_EC",
        "use_eem": False,
        "use_parafac": True,
        "features": [
            SS_COL,
            EC_COL,
        ]
    },

    {
        "name": "PARAFAC_TOC_SS_EC",
        "use_eem": False,
        "use_parafac": True,
        "features": [
            TOC_COL,
            SS_COL,
            EC_COL,
        ]
    },

    # =========================================================
    # EEM + PARAFAC
    # =========================================================
    {
        "name": "EEM",
        "use_eem": True,
        "use_parafac": False,
        "features": []
    },

    {
        "name": "EEM_TOC",
        "use_eem": True,
        "use_parafac": False,
        "features": [
            TOC_COL
        ]
    },

    {
        "name": "EEM_SS_EC",
        "use_eem": True,
        "use_parafac": False,
        "features": [
            SS_COL,
            EC_COL,
        ]
    },

    {
        "name": "EEM_TOC_SS_EC",
        "use_eem": True,
        "use_parafac": False,
        "features": [
            TOC_COL,
            SS_COL,
            EC_COL,
        ]
    },
    
    # =========================================================
    # EEM + PARAFAC
    # =========================================================
    {
        "name": "EEM_PARAFAC",
        "use_eem": True,
        "use_parafac": True,
        "features": []
    },

    {
        "name": "EEM_PARAFAC_TOC",
        "use_eem": True,
        "use_parafac": True,
        "features": [
            TOC_COL
        ]
    },

    {
        "name": "EEM_PARAFAC_SS_EC",
        "use_eem": True,
        "use_parafac": True,
        "features": [
            SS_COL,
            EC_COL,
        ]
    },

    {
        "name": "EEM_PARAFAC_TOC_SS_EC",
        "use_eem": True,
        "use_parafac": True,
        "features": [
            TOC_COL,
            SS_COL,
            EC_COL,
        ]
    },
    # =========================================================
    # EEM + PARAFAC
    # =========================================================
    {
        "name": "EEMpca",
        "use_eem": True,
        "use_parafac": False,
        "features": [],
        "eem_reduction": True,
    },

    {
        "name": "EEMpca_TOC",
        "use_eem": True,
        "use_parafac": False,
        "eem_reduction": "pca",
        "features": [
            TOC_COL
        ]
    },

    {
        "name": "EEMpca_SS_EC",
        "use_eem": True,
        "use_parafac": False,
        "eem_reduction": "pca",
        "features": [
            SS_COL,
            EC_COL,
        ]
    },

    {
        "name": "EEMpca_TOC_SS_EC",
        "use_eem": True,
        "use_parafac": False,
        "eem_reduction": "pca",
        "features": [
            TOC_COL,
            SS_COL,
            EC_COL,
        ]
    },
    
    # =========================================================
    # EEM + PARAFAC
    # =========================================================
    {
        "name": "EEMpca_PARAFAC",
        "use_eem": True,
        "use_parafac": True,
        "eem_reduction": "pca",
        "features": []
    },

    {
        "name": "EEMpca_PARAFAC_TOC",
        "use_eem": True,
        "use_parafac": True,
        "eem_reduction": "pca",
        "features": [
            TOC_COL
        ]
    },

    {
        "name": "EEMpca_PARAFAC_SS_EC",
        "use_eem": True,
        "use_parafac": True,
        "eem_reduction": "pca",
        "features": [
            SS_COL,
            EC_COL,
        ]
    },

    {
        "name": "EEMpca_PARAFAC_TOC_SS_EC",
        "use_eem": True,
        "use_parafac": True,
        "eem_reduction": "pca",
        "features": [
            TOC_COL,
            SS_COL,
            EC_COL,
        ]
    },
]

In [ ]:
# =========================================================
# MODELS
# =========================================================

MODELS = {

    "LinearRegression":
        LinearRegression(),

    "DecisionTreeRegressor":
        DecisionTreeRegressor(
            random_state=42
        ),

    "XGBRegressor":
        XGBRegressor(
            random_state=42
        )
}


# =========================================================
# RESULT DATAFRAME
# =========================================================

results = []


# =========================================================
# LOOP TARGETS
# =========================================================

for target in TARGETS_TO_RUN:

    print("\n" + "=" * 70)
    print("TARGET:", target)
    print("=" * 70)

    y_all = samples[
        target
    ].to_numpy(dtype=float)

    # =====================================================
    # VALID TARGETS INSIDE FIXED TRAIN / TEST SPLIT
    #
    # Feature values are assumed to contain no missing.
    # Only target NaN is removed.
    # =====================================================

    train_idx = train_idx_global[
        ~np.isnan(
            y_all[train_idx_global]
        )
    ]

    test_idx = test_idx_global[
        ~np.isnan(
            y_all[test_idx_global]
        )
    ]

    if (
        len(train_idx) < 10
        or len(test_idx) < 2
    ):
        print(
            "Skip target: too few valid samples"
        )
        continue

    y_train = y_all[
        train_idx
    ]

    y_test = y_all[
        test_idx
    ]

    print(
        "Valid train:",
        len(train_idx),
        "| Valid test:",
        len(test_idx)
    )


    # =====================================================
    # LOOP EXPERIMENTS
    # =====================================================

    for exp in EXPERIMENTS:

        exp_name = exp["name"]

        use_eem = exp.get(
            "use_eem",
            False
        )

        use_parafac = exp.get(
            "use_parafac",
            False
        )

        reduction = exp.get(
            "eem_reduction",
            None
        )


        # ---------------------------------------------
        # Remove target itself from features
        # ---------------------------------------------

        tab_features = [
            f
            for f in exp.get(
                "features",
                []
            )
            if f != target
        ]


        print(
            f"\nExperiment: {exp_name} | "
            f"EEM={use_eem} | "
            f"PARAFAC={use_parafac} | "
            f"features={tab_features} | "
            f"reduction={reduction}"
        )


        X_train_parts = []
        X_test_parts = []


        # =================================================
        # RAW / PCA EEM FEATURES
        # =================================================

        if use_eem:

            X_eem_train = X_eem_parafac[
                train_idx
            ]

            X_eem_test = X_eem_parafac[
                test_idx
            ]


            # Flatten
            X_eem_train = (
                X_eem_train.reshape(
                    X_eem_train.shape[0],
                    -1
                )
            )

            X_eem_test = (
                X_eem_test.reshape(
                    X_eem_test.shape[0],
                    -1
                )
            )


            # ---------------------------------------------
            # PCA
            # ---------------------------------------------

            if reduction == "pca":

                n_components = exp.get(
                    "n_components",
                    30
                )


                eem_scaler = StandardScaler()

                X_eem_train_scaled = (
                    eem_scaler.fit_transform(
                        X_eem_train
                    )
                )

                X_eem_test_scaled = (
                    eem_scaler.transform(
                        X_eem_test
                    )
                )


                max_components = min(
                    X_eem_train_scaled.shape[0],
                    X_eem_train_scaled.shape[1]
                )

                n_components = min(
                    n_components,
                    max_components
                )


                pca = PCA(
                    n_components=n_components,
                    random_state=42
                )


                X_eem_train_final = (
                    pca.fit_transform(
                        X_eem_train_scaled
                    )
                )

                X_eem_test_final = (
                    pca.transform(
                        X_eem_test_scaled
                    )
                )


                explained_variance = (
                    pca
                    .explained_variance_ratio_
                    .sum()
                )


                print(
                    f"PCA components: "
                    f"{n_components}"
                )

                print(
                    f"Explained variance: "
                    f"{explained_variance:.4f}"
                )

            else:

                X_eem_train_final = (
                    X_eem_train
                )

                X_eem_test_final = (
                    X_eem_test
                )


            X_train_parts.append(
                X_eem_train_final
            )

            X_test_parts.append(
                X_eem_test_final
            )


        # =================================================
        # PARAFAC FEATURES
        # =================================================

        if use_parafac:

            X_pf_train = (
                parafac_features[
                    train_idx
                ]
            )

            X_pf_test = (
                parafac_features[
                    test_idx
                ]
            )


            # Scale PARAFAC scores
            pf_scaler = StandardScaler()

            X_pf_train = (
                pf_scaler.fit_transform(
                    X_pf_train
                )
            )

            X_pf_test = (
                pf_scaler.transform(
                    X_pf_test
                )
            )


            X_train_parts.append(
                X_pf_train
            )

            X_test_parts.append(
                X_pf_test
            )


        # =================================================
        # TABULAR FEATURES
        # =================================================

        if len(tab_features) > 0:

            X_tab_train = (
                samples.loc[
                    train_idx,
                    tab_features
                ]
                .to_numpy(
                    dtype=float
                )
            )

            X_tab_test = (
                samples.loc[
                    test_idx,
                    tab_features
                ]
                .to_numpy(
                    dtype=float
                )
            )


            tab_scaler = StandardScaler()

            X_tab_train = (
                tab_scaler.fit_transform(
                    X_tab_train
                )
            )

            X_tab_test = (
                tab_scaler.transform(
                    X_tab_test
                )
            )


            X_train_parts.append(
                X_tab_train
            )

            X_test_parts.append(
                X_tab_test
            )


        # =================================================
        # CHECK INPUT
        # =================================================

        if len(X_train_parts) == 0:

            print(
                "Skip: no input features"
            )

            continue


        # =================================================
        # CONCATENATE FEATURES
        # =================================================

        X_train = np.hstack(
            X_train_parts
        )

        X_test = np.hstack(
            X_test_parts
        )


        print(
            "Train shape:",
            X_train.shape,
            "| Test shape:",
            X_test.shape
        )


        # =================================================
        # TRAIN MODELS
        # =================================================

        for model_name, base_model in MODELS.items():

            # fresh model every experiment
            model = clone(
                base_model
            )


            model.fit(
                X_train,
                y_train
            )


            pred = model.predict(
                X_test
            )


            mse = mean_squared_error(
                y_test,
                pred
            )


            result = {

                "target":
                    target,

                "experiment":
                    exp_name,

                "use_eem":
                    use_eem,

                "use_parafac":
                    use_parafac,

                "parafac_components":
                    (
                        PARAFAC_N_COMPONENTS
                        if use_parafac
                        else 0
                    ),

                "eem_reduction":
                    (
                        reduction
                        if reduction is not None
                        else "none"
                    ),

                "tabular_features":
                    ", ".join(
                        tab_features
                    ),

                "n_train":
                    len(y_train),

                "n_test":
                    len(y_test),

                "n_features":
                    X_train.shape[1],

                "model":
                    model_name,


                "R2":
                    r2_score(
                        y_test,
                        pred
                    ),

                "MSE":
                    mse,

                "MAE":
                    mean_absolute_error(
                        y_test,
                        pred
                    ),

                "MAPE":
                    mean_absolute_percentage_error(
                        y_test,
                        pred
                    ),

                "RMSE":
                    np.sqrt(mse)
            }


            results.append(
                result
            )


# =========================================================
# FINAL RESULTS
# =========================================================

results_df = pd.DataFrame(
    results
)

results_df

In [ ]:
results_df.to_csv("results.csv", index=False)

## Deep Learning

In [ ]:
import torch
import torch.nn as nn

class SimpleCNNFusion(nn.Module):

    def __init__(
        self,
        in_channels=1,
        n_tabular=0,
        dropout=0.3
    ):
        super().__init__()

        self.n_tabular = n_tabular

        # ==========================================
        # EEM branch
        # ==========================================
        self.cnn = nn.Sequential(

            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # ==========================================
        # Tabular branch
        # ==========================================
        if n_tabular > 0:

            self.tabular = nn.Sequential(
                nn.Linear(n_tabular, 32),
                nn.ReLU(),

                nn.Linear(32, 16),
                nn.ReLU()
            )

            fusion_dim = 128 + 16

        else:
            self.tabular = None

            fusion_dim = 128

        # ==========================================
        # Regressor
        # ==========================================
        self.regressor = nn.Sequential(

            nn.Linear(
                fusion_dim,
                64
            ),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(
                64,
                1
            )
        )

    def forward(
        self,
        eem,
        tabular=None
    ):

        # EEM
        x_img = self.cnn(eem)
        x_img = self.global_pool(x_img)
        x_img = torch.flatten(x_img, 1)

        # EEM only
        if self.tabular is None:

            x = x_img

        # EEM + tabular
        else:

            x_tab = self.tabular(
                tabular
            )

            x = torch.cat(
                [x_img, x_tab],
                dim=1
            )

        return self.regressor(
            x
        ).squeeze(1)

import torch
import torch.nn as nn


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += identity
        out = self.relu(out)

        return out


class ResNetEEMBackbone(nn.Module):

    def __init__(self, layers, in_channels=1):
        super().__init__()

        self.in_channels = 64

        # input: EEM or EEM + FFT
        self.conv1 = nn.Conv2d(
            in_channels,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(
            64, layers[0], stride=1
        )
        self.layer2 = self._make_layer(
            128, layers[1], stride=2
        )
        self.layer3 = self._make_layer(
            256, layers[2], stride=2
        )
        self.layer4 = self._make_layer(
            512, layers[3], stride=2
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def _make_layer(self, out_channels, blocks, stride):

        layers = [
            BasicBlock(
                self.in_channels,
                out_channels,
                stride
            )
        ]

        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(
                BasicBlock(
                    self.in_channels,
                    out_channels
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):

        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)

        return torch.flatten(x, 1)


def get_resnet10_backbone(in_channels=1):
    return ResNetEEMBackbone(
        layers=[1, 1, 1, 1],
        in_channels=in_channels
    )


def get_resnet18_backbone(in_channels=1):
    return ResNetEEMBackbone(
        layers=[2, 2, 2, 2],
        in_channels=in_channels
    )

class ResNetFusion(nn.Module):

    def __init__(
        self,
        backbone,
        n_tabular=0,
        tab_hidden=16,
        fusion_hidden=128,
        dropout=0.3
    ):
        super().__init__()

        self.backbone = backbone

        if n_tabular > 0:
            self.tabular_branch = nn.Sequential(
                nn.Linear(n_tabular, tab_hidden),
                nn.ReLU(),
                nn.Linear(tab_hidden, tab_hidden),
                nn.ReLU()
            )

            fusion_dim = 512 + tab_hidden

        else:
            self.tabular_branch = None
            fusion_dim = 512

        self.regressor = nn.Sequential(
            nn.Linear(fusion_dim, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1)
        )

    def forward(self, eem, tabular=None):

        z_eem = self.backbone(eem)

        if self.tabular_branch is None:
            z = z_eem

        else:
            if tabular is None:
                raise ValueError(
                    "tabular input is required when n_tabular > 0"
                )

            z_tab = self.tabular_branch(tabular)

            z = torch.cat(
                [z_eem, z_tab],
                dim=1
            )

        return self.regressor(z).squeeze(1)

def get_resnet10_fusion(
    in_channels=1,
    n_tabular=0
):
    return ResNetFusion(
        backbone=get_resnet10_backbone(
            in_channels=in_channels
        ),
        n_tabular=n_tabular,
        dropout=0.3
    )


def get_resnet18_fusion(
    in_channels=1,
    n_tabular=0
):
    return ResNetFusion(
        backbone=get_resnet18_backbone(
            in_channels=in_channels
        ),
        n_tabular=n_tabular,
        dropout=0.3
    )

In [ ]:
from torch.utils.data import Dataset, DataLoader

class EEMDataset(Dataset):

    def __init__(
        self,
        X_eem,
        X_tab,
        y
    ):

        self.X_eem = torch.tensor(
            X_eem,
            dtype=torch.float32
        )

        self.X_tab = torch.tensor(
            X_tab,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):

        return (
            self.X_eem[idx],
            self.X_tab[idx],
            self.y[idx]
        )

In [ ]:
import copy

import copy
import numpy as np
import torch
import torch.nn as nn


def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=100,
    lr=1e-3,
    patience=15
):

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    best_val_loss = np.inf
    best_state = None

    epochs_no_improve = 0

    for epoch in range(epochs):

        # =========================
        # Train
        # =========================
        model.train()

        train_loss = 0.0

        for eem, tab, y in train_loader:

            eem = eem.to(device)
            tab = tab.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            pred = model(
                eem,
                tab
            )

            loss = criterion(
                pred,
                y
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item()
                * len(y)
            )

        train_loss /= len(
            train_loader.dataset
        )

        # =========================
        # Validation
        # =========================
        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for eem, tab, y in val_loader:

                eem = eem.to(device)
                tab = tab.to(device)
                y = y.to(device)

                pred = model(
                    eem,
                    tab
                )

                loss = criterion(
                    pred,
                    y
                )

                val_loss += (
                    loss.item()
                    * len(y)
                )

        val_loss /= len(
            val_loader.dataset
        )

        # =========================
        # Check improvement
        # =========================
        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_no_improve = 0

        else:

            epochs_no_improve += 1

        # =========================
        # Print
        # =========================
        if (epoch + 1) % 10 == 0:

            print(
                f"Epoch {epoch+1:03d} | "
                f"Train: {train_loss:.4f} | "
                f"Val: {val_loss:.4f} | "
                f"No improve: {epochs_no_improve}"
            )

        # =========================
        # Early stopping
        # =========================
        if epochs_no_improve >= patience:

            print(
                f"Early stopping at epoch "
                f"{epoch + 1}"
            )

            break

    # Restore best model
    model.load_state_dict(
        best_state
    )

    return model

def predict_model(
    model,
    loader,
    device
):

    model.eval()

    preds = []
    ys = []

    with torch.no_grad():

        for eem, tab, y in loader:

            eem = eem.to(device)
            tab = tab.to(device)

            pred = model(
                eem,
                tab
            )

            preds.append(
                pred.cpu().numpy()
            )

            ys.append(
                y.numpy()
            )

    return (
        np.concatenate(ys),
        np.concatenate(preds)
    )

In [ ]:
import numpy as np
import pandas as pd

BOD_COL = "BOD\n(0.0)"
COD_COL = "COD\n(0.0)"
TOC_COL = "TOC\n(0.0)"

EC_COL = "EC\n(0)"
SS_COL = "SS\n(0.0)"

FEATURE_SETS = {
    "EEM_only": [],

    "EC": [
        EC_COL
    ],

    "SS": [
        SS_COL
    ],

    "EC_SS": [
        EC_COL,
        SS_COL
    ],

}

In [ ]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from torch.utils.data import DataLoader


# =========================
# Config
# =========================
TARGET = BOD_COL
TAB_FEATURES = [EC_COL, SS_COL]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# Valid samples
# =========================
mask = samples[[TARGET] + TAB_FEATURES].notna().all(axis=1).to_numpy()
mask &= ~np.isnan(X_eem_fft.reshape(len(X_eem_fft), -1)).any(axis=1)

y = samples.loc[mask, TARGET].to_numpy(np.float32)
X_tab = samples.loc[mask, TAB_FEATURES].to_numpy(np.float32)
X_img = X_eem_fft[mask].astype(np.float32)   # (N, 2, 57, 271)


# =========================
# Split
# =========================
idx = np.arange(len(y))

train_val_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=42
)

train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.2, random_state=42
)


# =========================
# Normalize EEM + FFT
# separately for each channel
# =========================
mean = X_img[train_idx].mean(axis=(0, 2, 3), keepdims=True)
std = X_img[train_idx].std(axis=(0, 2, 3), keepdims=True) + 1e-8

X_img = (X_img - mean) / std


# =========================
# Scale tabular
# =========================
tab_scaler = StandardScaler()

X_tab_scaled = X_tab.copy()

X_tab_scaled[train_idx] = tab_scaler.fit_transform(X_tab[train_idx])
X_tab_scaled[val_idx] = tab_scaler.transform(X_tab[val_idx])
X_tab_scaled[test_idx] = tab_scaler.transform(X_tab[test_idx])


# =========================
# Scale target
# =========================
y_scaler = StandardScaler()

y_scaled = y.copy()

y_scaled[train_idx] = y_scaler.fit_transform(
    y[train_idx, None]
).ravel()

y_scaled[val_idx] = y_scaler.transform(
    y[val_idx, None]
).ravel()

y_scaled[test_idx] = y_scaler.transform(
    y[test_idx, None]
).ravel()


# =========================
# DataLoader
# =========================
def make_loader(indices, shuffle=False):
    ds = EEMDataset(
        X_eem=X_img[indices],
        # X_tab_scaled[indices],
        y=y_scaled[indices]
    )

    return DataLoader(
        ds,
        batch_size=32,
        shuffle=shuffle
    )


train_loader = make_loader(train_idx, True)
val_loader = make_loader(val_idx)
test_loader = make_loader(test_idx)


# =========================
# ResNet10: 2 input channels
# =========================
model = ResNetFusion(
    backbone=get_resnet10_backbone(in_channels=2),
    n_tabular=2,
    dropout=0.3
).to(device)


# =========================
# Train
# =========================
model = train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=200,
    lr=1e-3,
    patience=20
)


# =========================
# Predict
# =========================
y_test_scaled, pred_scaled = predict_model(
    model,
    test_loader,
    device
)

y_test = y_scaler.inverse_transform(
    y_test_scaled[:, None]
).ravel()

pred = y_scaler.inverse_transform(
    pred_scaled[:, None]
).ravel()


# =========================
# Metrics
# =========================
mse = mean_squared_error(y_test, pred)

print("R2  :", r2_score(y_test, pred))
print("RMSE:", np.sqrt(mse))
print("MAE :", mean_absolute_error(y_test, pred))
print("MAPE:", mean_absolute_percentage_error(y_test, pred))

In [ ]:
import numpy as np
import pandas as pd

from scipy.optimize import nnls

from tensorly.decomposition import non_negative_parafac

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


def project_to_parafac(X, B_ex, C_em):
    """
    Project new EEM samples onto fixed PARAFAC spectral components.

    Parameters
    ----------
    X : (N, Ex, Em)
    B_ex : (Ex, rank)
    C_em : (Em, rank)

    Returns
    -------
    Z : (N, rank)
        PARAFAC component scores for each sample.
    """

    rank = B_ex.shape[1]

    # Each column is one PARAFAC component:
    # outer(excitation loading, emission loading)
    basis = np.stack([
        np.outer(B_ex[:, r], C_em[:, r]).ravel()
        for r in range(rank)
    ], axis=1)  # (Ex*Em, rank)

    Z = []

    for x in X:
        x_flat = x.ravel()

        # Non-negative least squares
        score, _ = nnls(basis, x_flat)

        Z.append(score)

    return np.asarray(Z)


def parafac_predict_wqp(
    EEM,
    df_samples,
    target_col,
    n_components=5,
    test_size=0.2,
    random_state=42,
    group_col=None
):
    """
    PARAFAC feature extraction + SVR regression
    for one water-quality parameter.
    """

    # ----------------------------------------------------
    # 1. Prepare target
    # ----------------------------------------------------

    y = pd.to_numeric(
        df_samples[target_col],
        errors="coerce"
    ).values

    # Remove samples with missing target
    valid = np.isfinite(y)

    X = EEM[valid].astype(np.float64)
    y = y[valid]

    metadata = df_samples.loc[valid].copy()

    print("Valid samples:", len(y))
    print("EEM shape:", X.shape)

    # ----------------------------------------------------
    # 2. Check EEM
    # ----------------------------------------------------

    if not np.isfinite(X).all():
        raise ValueError(
            "EEM contains NaN/Inf. "
            "Handle scatter/missing regions before PARAFAC."
        )

    # Non-negative PARAFAC requires non-negative values
    if X.min() < 0:
        print(
            "Warning: negative EEM values found. "
            "Clipping them to zero."
        )
        X = np.clip(X, 0, None)

    # ----------------------------------------------------
    # 3. Train/test split
    # ----------------------------------------------------

    indices = np.arange(len(X))

    if group_col is None:

        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_size,
            random_state=random_state
        )

    else:

        groups = metadata[group_col].values

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=random_state
        )

        train_idx, test_idx = next(
            splitter.split(X, y, groups)
        )

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    print("Train:", X_train.shape)
    print("Test :", X_test.shape)

    # ----------------------------------------------------
    # 4. Fit PARAFAC ONLY on training EEMs
    # ----------------------------------------------------

    cp_model = non_negative_parafac(
        X_train,
        rank=n_components,
        init="svd",
        n_iter_max=500,
        tol=1e-7,
        random_state=random_state
    )

    weights, factors = cp_model

    A_train, B_ex, C_em = factors

    # weights may theoretically be None
    if weights is None:
        weights = np.ones(n_components)

    # Sample-mode scores
    Z_train = A_train * weights[None, :]

    print("PARAFAC train features:", Z_train.shape)

    # ----------------------------------------------------
    # 5. Project TEST EEM onto TRAIN PARAFAC components
    # ----------------------------------------------------

    Z_test = project_to_parafac(
        X_test,
        B_ex,
        C_em
    )

    print("PARAFAC test features:", Z_test.shape)

    # ----------------------------------------------------
    # 6. Regression model
    # ----------------------------------------------------

    regressor = make_pipeline(
        StandardScaler(),
        SVR(
            kernel="rbf",
            C=10,
            epsilon=0.1,
            gamma="scale"
        )
    )

    # Standardize y during training
    model = TransformedTargetRegressor(
        regressor=regressor,
        transformer=StandardScaler()
    )

    model.fit(Z_train, y_train)

    # ----------------------------------------------------
    # 7. Prediction
    # ----------------------------------------------------

    pred = model.predict(Z_test)

    # ----------------------------------------------------
    # 8. Metrics
    # ----------------------------------------------------

    r2 = r2_score(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, pred)

    print("\n===== RESULTS =====")
    print(f"Target: {target_col}")
    print(f"PARAFAC components: {n_components}")
    print(f"R2   : {r2:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")

    # ----------------------------------------------------
    # 9. Prediction dataframe
    # ----------------------------------------------------

    result_df = metadata.iloc[test_idx].copy()

    result_df["y_true"] = y_test
    result_df["y_pred"] = pred

    return {
        "model": model,
        "parafac_model": cp_model,

        "Z_train": Z_train,
        "Z_test": Z_test,

        "excitation_loadings": B_ex,
        "emission_loadings": C_em,

        "y_train": y_train,
        "y_test": y_test,
        "pred": pred,

        "results": result_df,

        "metrics": {
            "R2": r2,
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae
        }
    }

In [ ]:
TARGETS_TO_RUN

In [ ]:
# results = {}
# for target in TARGETS_TO_RUN:
#     result = parafac_predict_wqp(
#         EEM=eem,
#         df_samples=samples,
#         target_col=target,
#         n_components=5
#     )
#     results[target] = result

In [ ]:
# for target, result in results.items():
#     print(target)
#     print(result["metrics"])